# 02 — Source-level outer and inner splits

Splits are assigned on a table with exactly one row per source video and
expanded to sequences afterward. Every clip, mirror, and augmentation from a
video inherits that video's fold. Outer-test sources are never used for
encoder training, checkpoint selection, read-out tuning, or preprocessing
statistics. Inner folds tune read-outs using outer-training sources only.

Dataset annotations keep source counts reasonably balanced across folds and
later support one explicitly named confounding control; they are not diagnoses
or primary prediction targets. Multiple videos
may still depict the same unidentified person, so these folds establish
held-out-video—not held-out-person—evaluation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
from laterality.data import load_cohort
from laterality.splitting import build_source_splits, save_splits
from laterality.visualization import split_figure

cohort = load_cohort(context)
split_config = context.protocol["splits"]
splits = build_source_splits(
    cohort.table,
    context.protocol["data"]["conditions"],
    outer_folds=split_config["outer_folds"],
    inner_folds=split_config["inner_folds"],
    seed=split_config["seed"],
)
split_artifact = save_splits(context, cohort, splits)

fold_summary = [
    {
        "fold": fold["fold"],
        "train_sources": len(fold["train_sources"]),
        "test_sources": len(fold["test_sources"]),
        "train_source_counts": fold["train_source_counts"],
        "test_source_counts": fold["test_source_counts"],
        "inner_folds": len(fold["inner_readout_folds"]),
    }
    for fold in splits["folds"]
]
show_inline(split_figure(context, splits))
{
    "split_artifact": str(split_artifact),
    "source_census": splits["source_census"],
    "folds": fold_summary,
}